# ДЗ №4. Соревнование на Kaggle

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

train = pd.read_parquet('data/train.pq')
items = pd.read_parquet('data/items.pq')
test_users = pd.read_csv('data/test_users.csv')

train['timestamp'] = pd.to_datetime(train['timestamp'])
train = train.sort_values('timestamp').reset_index(drop=True)
max_time = train['timestamp'].max()

target_cutoff = max_time - pd.Timedelta(days=7)
train_data = train[train['timestamp'] < target_cutoff].copy()

print(f"Training data: {train_data.shape}")

user_ids = train_data['user_id'].unique()
item_ids = train_data['item_id'].unique()
user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
item_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}
idx_to_item = {idx: iid for iid, idx in item_to_idx.items()}

train_data['weight'] = 1.0
train_data.loc[train_data['rating'] >= 3, 'weight'] = 3.0
train_data.loc[train_data['rating'] >= 4, 'weight'] = 5.0
train_data.loc[train_data['is_purchased'] == True, 'weight'] = 10.0

interactions = train_data.groupby(['user_id', 'item_id'])['weight'].sum().reset_index()
row = interactions['user_id'].map(user_to_idx).values
col = interactions['item_id'].map(item_to_idx).values
data = interactions['weight'].values
user_item_matrix = csr_matrix((data, (row, col)), shape=(len(user_ids), len(item_ids)))

print(f"Matrix: {user_item_matrix.shape}, non-zero: {user_item_matrix.nnz}")

print("Training ALS 1...")
als1 = AlternatingLeastSquares(factors=200, regularization=0.1, iterations=40, num_threads=-1, random_state=42)
als1.fit(user_item_matrix)

print("Training ALS 2...")
als2 = AlternatingLeastSquares(factors=150, regularization=0.2, iterations=30, num_threads=-1, random_state=123)
als2.fit(user_item_matrix)

print("Training ALS 3...")
als3 = AlternatingLeastSquares(factors=100, regularization=0.05, iterations=25, num_threads=-1, random_state=456)
als3.fit(user_item_matrix)

print("Building content features...")
item_categories = items.set_index('item_id') if 'item_id' in items.columns else items

user_category_prefs = defaultdict(lambda: defaultdict(float))
item_tags = {}

for _, row in train_data[train_data['is_purchased'] == True].iterrows():
    uid = row['user_id']
    iid = row['item_id']
    w = row['weight']
    
    if iid in item_categories.index:
        cats = item_categories.loc[iid, 'category_tags']
        if isinstance(cats, list):
            for tag in cats:
                user_category_prefs[uid][f"tag_{tag}"] += w
                if iid not in item_tags:
                    item_tags[iid] = set()
                item_tags[iid].add(f"tag_{tag}")

print("Computing popularity scores...")
global_pop = train_data['item_id'].value_counts().head(100).index.tolist()
recent = train_data[train_data['timestamp'] > train_data['timestamp'].max() - pd.Timedelta(days=7)]
recent_pop = recent['item_id'].value_counts().head(100).index.tolist()
purchase_rates = train_data.groupby('item_id')['is_purchased'].mean().to_dict()

print("Generating recommendations...")
popular_items = train_data['item_id'].value_counts().head(20).index.tolist()
user_to_idx_all = {uid: idx for idx, uid in enumerate(user_ids)}

recommendations = []

for i, user_id in enumerate(test_users['user_id']):
    if i % 2000 == 0:
        print(f"  {i}/{len(test_users)} ({100*i/len(test_users):.1f}%)")
    
    if user_id not in user_to_idx_all:
        for item_id in popular_items:
            recommendations.append({'user_id': user_id, 'item_id': item_id})
        continue
    
    user_idx = user_to_idx_all[user_id]
    user_items = user_item_matrix[user_idx]
    
    try:
        als1_recs = als1.recommend(user_idx, user_items, N=100, filter_already_liked_items=True)
        als2_recs = als2.recommend(user_idx, user_items, N=100, filter_already_liked_items=True)
        als3_recs = als3.recommend(user_idx, user_items, N=100, filter_already_liked_items=True)
    except:
        for item_id in popular_items:
            recommendations.append({'user_id': user_id, 'item_id': item_id})
        continue
    
    scores = defaultdict(float)
    user_prefs = user_category_prefs.get(user_id, {})
    
    for item_idx, score in zip(als1_recs[0], als1_recs[1]):
        item_id = idx_to_item[item_idx]
        scores[item_id] += score * 1.0
    
    for item_idx, score in zip(als2_recs[0], als2_recs[1]):
        item_id = idx_to_item[item_idx]
        scores[item_id] += score * 0.8
    
    for item_idx, score in zip(als3_recs[0], als3_recs[1]):
        item_id = idx_to_item[item_idx]
        scores[item_id] += score * 0.6
    
    for item_id in list(scores.keys())[:50]:
        if item_id in item_tags and user_prefs:
            item_cats = item_tags.get(item_id, set())
            if item_cats:
                overlap = len(item_cats & set(user_prefs.keys()))
                scores[item_id] *= (1.0 + 0.1 * overlap)
        
        scores[item_id] *= (0.5 + 0.5 * purchase_rates.get(item_id, 0))
        
        if item_id in recent_pop:
            scores[item_id] *= 1.2
        if item_id in global_pop:
            scores[item_id] *= 1.1
    
    sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_items = [item_id for item_id, _ in sorted_items[:20]]
    
    if len(top_items) < 20:
        for item_id in popular_items:
            if item_id not in top_items:
                top_items.append(item_id)
            if len(top_items) >= 20:
                break
    
    for item_id in top_items[:20]:
        recommendations.append({'user_id': user_id, 'item_id': item_id})

submission = pd.DataFrame(recommendations)

missing = set(test_users['user_id']) - set(submission['user_id'])
if missing:
    extra = [{'user_id': u, 'item_id': i} for u in missing for i in popular_items]
    submission = pd.concat([submission, pd.DataFrame(extra)])

submission = submission.groupby('user_id').head(20).reset_index(drop=True)

user_counts = submission.groupby('user_id').size()
if (user_counts < 20).any():
    extra = []
    for u in user_counts[user_counts < 20].index:
        cur = submission[submission['user_id'] == u]['item_id'].tolist()
        need = 20 - len(cur)
        for iid in popular_items:
            if iid not in cur and need > 0:
                extra.append({'user_id': u, 'item_id': iid})
                need -= 1
    submission = pd.concat([submission, pd.DataFrame(extra)])
    submission = submission.groupby('user_id').head(20).reset_index(drop=True)

submission.to_csv('submission.csv', index=False)
print(f"\nSaved: {submission.shape}")
print(f"Users: {submission['user_id'].nunique()}")
print(f"Unique items: {submission['item_id'].nunique()}")

/Users/maksimlunin/Desktop/jupyter/Рекомендательные системы/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training data: (11296067, 6)
Matrix: (347141, 31163), non-zero: 11296067
Training ALS 1...


100%|██████████| 40/40 [10:03<00:00, 15.10s/it]


Training ALS 2...


100%|██████████| 30/30 [04:59<00:00,  9.97s/it]


Training ALS 3...


100%|██████████| 25/25 [02:39<00:00,  6.37s/it]


Building content features...
Computing popularity scores...
Generating recommendations...
  0/185282 (0.0%)
  2000/185282 (1.1%)
  4000/185282 (2.2%)
  6000/185282 (3.2%)
  8000/185282 (4.3%)
  10000/185282 (5.4%)
  12000/185282 (6.5%)
  14000/185282 (7.6%)
  16000/185282 (8.6%)
  18000/185282 (9.7%)
  20000/185282 (10.8%)
  22000/185282 (11.9%)
  24000/185282 (13.0%)
  26000/185282 (14.0%)
  28000/185282 (15.1%)
  30000/185282 (16.2%)
  32000/185282 (17.3%)
  34000/185282 (18.4%)
  36000/185282 (19.4%)
  38000/185282 (20.5%)
  40000/185282 (21.6%)
  42000/185282 (22.7%)
  44000/185282 (23.7%)
  46000/185282 (24.8%)
  48000/185282 (25.9%)
  50000/185282 (27.0%)
  52000/185282 (28.1%)
  54000/185282 (29.1%)
  56000/185282 (30.2%)
  58000/185282 (31.3%)
  60000/185282 (32.4%)
  62000/185282 (33.5%)
  64000/185282 (34.5%)
  66000/185282 (35.6%)
  68000/185282 (36.7%)
  70000/185282 (37.8%)
  72000/185282 (38.9%)
  74000/185282 (39.9%)
  76000/185282 (41.0%)
  78000/185282 (42.1%)
  80000/